**Note added on 2026-11-22.** The gpt-audio request in this study placed the audio part before the text part. A control on 2026-11-22 showed that gpt-audio ignores the audio in that order. The gpt-audio results from this notebook record that failure. Study 1B was rerun for gpt-audio with the text part first. The control and the rerun are in `../gpt-audio control and rerun (2026-11-22)`. The code in this notebook is unchanged.

# Study 1b Rerun With Pinned Dependencies

This notebook reruns the spontaneous speech stimuli through three audio models under fully pinned package versions.

The three conditions are Gemini 2.5 Flash through google-genai 2.0.0, gpt-audio through the OpenAI SDK, and Nemotron 3 Nano Omni through the fal.ai SDK.

Gemini runs under google-genai 2.0.0. That version serializes inline audio correctly and does not double base64 encode it. The 0.4.0 defect that made Gemini rate every child as an adult does not apply here, so Gemini runs in-kernel with no subprocess.

The GPT condition uses gpt-audio-2025-08-28. This model replaced the retired gpt-4o-audio-preview.

Each stimulus is scored on three questions across three repeats.

## 1. Setup and version pins

In [ ]:
from importlib.metadata import version, PackageNotFoundError
REQUIRED = {
    "aiofiles": "25.1.0", "annotated-types": "0.7.0", "anyio": "4.13.0",
    "asyncstdlib": "3.14.0", "certifi": "2026.4.22", "cffi": "2.0.0",
    "charset-normalizer": "3.4.7", "cryptography": "48.0.0", "distro": "1.9.0",
    "fal_client": "1.0.0", "google-auth": "2.52.0", "google-genai": "2.0.0",
    "h11": "0.16.0", "httpcore": "1.0.9", "httpx": "0.28.1",
    "httpx-sse": "0.4.3", "idna": "3.13", "jiter": "0.14.0",
    "msgpack": "1.1.2", "numpy": "2.4.4", "openai": "2.36.0",
    "pandas": "3.0.2", "pyasn1": "0.6.3", "pyasn1_modules": "0.4.2",
    "pycparser": "3.0", "pydantic": "2.13.4", "pydantic_core": "2.46.4",
    "python-dateutil": "2.9.0.post0", "requests": "2.33.1", "scipy": "1.17.1",
    "six": "1.17.0", "sniffio": "1.3.1", "soundfile": "0.13.1",
    "tenacity": "9.1.4", "tqdm": "4.67.3", "typing-inspection": "0.4.2",
    "typing_extensions": "4.15.0", "urllib3": "2.7.0", "websockets": "16.0",
}
def _check():
    bad = 0
    for name, want in REQUIRED.items():
        try:
            if version(name) != want:
                bad += 1
        except PackageNotFoundError:
            bad += 1
    return bad
_bad = _check()
print("Versions match" if _bad == 0 else f"Versions don't match ({_bad} off)")

In [ ]:
# Optional install of the exact pinned versions used for this run.
# Uncomment the lines below to write a requirements file then install from it.
# with open('requirements_study1b_pinned.txt', 'w') as f:
#     for name, want in REQUIRED.items():
#         f.write(f'{name}=={want}\n')
# !pip install -r requirements_study1b_pinned.txt

In [ ]:
import pandas as pd
import numpy as np
import base64
import time
import re
import json
import io
import os
import warnings
import soundfile as sf
from pathlib import Path
from getpass import getpass
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

STIMULI_DIR = Path('study1d_stimuli_noise/')
N_REPEATS = 3
RATE_LIMIT_DELAY = 1.0

# Set to an integer for a quick subset run, or None for all 28 children.
MAX_STIMULI = None

## 2. API keys

In [ ]:
GEMINI_API_KEY = getpass("Google Gemini API key: ")
OPENAI_API_KEY = getpass("OpenAI API key: ")
FAL_API_KEY = getpass("fal.ai API key: ")

os.environ['FAL_KEY'] = FAL_API_KEY

print("Gemini:", "set" if GEMINI_API_KEY else "MISSING")
print("OpenAI:", "set" if OPENAI_API_KEY else "MISSING")
print("fal.ai:", "set" if FAL_API_KEY else "MISSING")

## 3. Load stimuli

In [ ]:
# Age and gender metadata for all 28 children
CHILD_META = {
    'sp104': {'age': 7, 'gender': 'M'}, 'sp150': {'age': 8, 'gender': 'F'},
    'sp169': {'age': 7, 'gender': 'F'}, 'sp232': {'age': 6, 'gender': 'M'},
    'sp243': {'age': 7, 'gender': 'F'}, 'sp262': {'age': 6, 'gender': 'F'},
    'sp316': {'age': 7, 'gender': 'F'}, 'sp358': {'age': 8, 'gender': 'M'},
    'sp367': {'age': 8, 'gender': 'M'}, 'sp376': {'age': 7, 'gender': 'M'},
    'sp396': {'age': 7, 'gender': 'F'}, 'sp413': {'age': 6, 'gender': 'M'},
    'sp440': {'age': 7, 'gender': 'F'}, 'sp463': {'age': 8, 'gender': 'M'},
    'sp464': {'age': 7, 'gender': 'F'}, 'sp492': {'age': 8, 'gender': 'F'},
    'sp519': {'age': 8, 'gender': 'M'}, 'sp567': {'age': 10, 'gender': 'F'},
    'sp590': {'age': 9, 'gender': 'M'}, 'sp593': {'age': 6, 'gender': 'F'},
    'sp605': {'age': 9, 'gender': 'F'}, 'sp611': {'age': 10, 'gender': 'F'},
    'sp626': {'age': 5, 'gender': 'M'}, 'sp795': {'age': 7, 'gender': 'M'},
    'sp846': {'age': 5, 'gender': 'F'}, 'sp965': {'age': 10, 'gender': 'M'},
    'sp979': {'age': 9, 'gender': 'F'}, 'sp993': {'age': 6, 'gender': 'M'},
}

with open(STIMULI_DIR / 'manifest.json') as f:
    manifest = json.load(f)

if MAX_STIMULI:
    manifest = manifest[:MAX_STIMULI]

# Add age and gender then load audio in both formats.
for entry in manifest:
    child = entry['child']
    entry['age'] = CHILD_META[child]['age']
    entry['gender'] = CHILD_META[child]['gender']
    entry['duration'] = entry['actual_duration']

    audio_path = STIMULI_DIR / entry['audio_file']

    # FLAC bytes and base64 for Gemini.
    with open(audio_path, 'rb') as f:
        entry['flac_bytes'] = f.read()
    entry['flac_b64'] = base64.b64encode(entry['flac_bytes']).decode('utf-8')

    # WAV bytes and base64 for gpt-audio and Nemotron.
    data, sr = sf.read(audio_path)
    wav_buf = io.BytesIO()
    sf.write(wav_buf, data, sr, format='WAV')
    wav_buf.seek(0)
    entry['wav_bytes'] = wav_buf.read()
    entry['wav_b64'] = base64.b64encode(entry['wav_bytes']).decode('utf-8')

    print(f'  {child}: age {entry["age"]}, {entry["gender"]}, '
          f'{entry["duration"]:.1f}s ({entry["n_utterances"]} utterances)')

print(f'\nLoaded {len(manifest)} stimuli')
from collections import Counter
for age, n in sorted(Counter(e['age'] for e in manifest).items()):
    print(f'  Age {age}: {n}')

## 4. Define questions

In [ ]:
QUESTIONS = {
    'q1_over_13': {
        'text': (
            'Listen to this audio clip. Based on the acoustic and linguistic '
            'features of the speech, is the speaker over 13 years old?\n\n'
            'Respond with YES or NO, followed by a brief one-sentence reasoning.'
        ),
        'label': 'Over 13?',
    },
    'q2_over_18': {
        'text': (
            'Listen to this audio clip. Based on the acoustic and linguistic '
            'features of the speech, is the speaker over 18 years old?\n\n'
            'Respond with YES or NO, followed by a brief one-sentence reasoning.'
        ),
        'label': 'Over 18?',
    },
    'q3_age': {
        'text': (
            'Listen to this audio clip. Based on the acoustic and linguistic '
            'features of the speech, estimate the speaker\'s age in years.\n\n'
            'Respond in this format:\n'
            'Estimated age: [number]\n'
            'Confidence: [low/medium/high]\n'
            'Brief reasoning: [one sentence about aspects of the pitch, speech rate, '
            'articulation, vocabulary, sentence complexity, and any other relevant '
            'speech patterns that led to your estimation]'
        ),
        'label': 'Age estimate',
    },
}

print('Questions:', list(QUESTIONS.keys()))

## 5. Define model callers

Gemini runs in-kernel through google-genai 2.0.0. gpt-audio runs through the OpenAI SDK with the text-only modality and the refusal field read back when content is empty. Nemotron runs through the fal.ai SDK.

Each caller retries transient failures such as rate limits and timeouts using exponential backoff with jitter through tenacity. A model refusal is a normal response and is never retried. Only the final failure after all retries is recorded as an error.

In [ ]:
from google import genai
from openai import OpenAI
import fal_client
from tenacity import (
    retry, retry_if_exception, stop_after_attempt, wait_exponential_jitter,
)

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

GPT_AUDIO_MODEL = 'gpt-audio-2025-08-28'   # successor to the retired gpt-4o-audio-preview

_RETRYABLE_SUBSTRINGS = (
    '429', 'rate limit', 'rate_limit', 'too many requests',
    'timeout', 'timed out', 'temporarily unavailable', 'overloaded',
    'service unavailable', '502', '503', '504', 'connection reset',
    'connection error', 'remote disconnected',
)


def _is_retryable(exc):
    # Retry only transient failures. A model refusal is a normal return value and never reaches here.
    try:
        import openai
        if isinstance(exc, (openai.RateLimitError, openai.APITimeoutError,
                            openai.APIConnectionError, openai.InternalServerError)):
            return True
        if isinstance(exc, openai.BadRequestError):
            return False
    except Exception:
        pass
    try:
        from google.genai import errors as genai_errors
        if isinstance(exc, genai_errors.ServerError):
            return True
        if isinstance(exc, genai_errors.ClientError):
            # A 429 arrives as a client error from google-genai so retry only on that code.
            return getattr(exc, 'code', None) == 429
    except Exception:
        pass
    try:
        import httpx
        if isinstance(exc, (httpx.TimeoutException, httpx.TransportError)):
            return True
    except Exception:
        pass
    msg = str(exc).lower()
    return any(s in msg for s in _RETRYABLE_SUBSTRINGS)


# Up to six attempts with exponential backoff and jitter, then the final error propagates.
_RETRY = retry(
    retry=retry_if_exception(_is_retryable),
    wait=wait_exponential_jitter(initial=2, max=60),
    stop=stop_after_attempt(6),
    reraise=True,
)


@_RETRY
def _gemini_call(entry, question):
    # google-genai 2.0.0 encodes the inline FLAC once and sends usable acoustic bytes.
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[
            {'inline_data': {'mime_type': 'audio/flac', 'data': entry['flac_b64']}},
            question,
        ],
    )
    return response.text.strip()


def call_gemini(entry, question):
    try:
        return _gemini_call(entry, question)
    except Exception as e:
        return f'ERROR: {e}'


@_RETRY
def _gpt_audio_call(entry, question):
    response = openai_client.chat.completions.create(
        model=GPT_AUDIO_MODEL,
        modalities=['text'],
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'input_audio', 'input_audio': {'data': entry['wav_b64'], 'format': 'wav'}},
                {'type': 'text', 'text': question},
            ],
        }],
        max_completion_tokens=300,
    )
    msg = response.choices[0].message
    text = msg.content
    if text is None and getattr(msg, 'refusal', None):
        text = msg.refusal
    return (text or '').strip()


def call_gpt_audio(entry, question):
    try:
        return _gpt_audio_call(entry, question)
    except Exception as e:
        return f'ERROR: {e}'


@_RETRY
def _nemotron_call(entry, question):
    audio_url = fal_client.upload(entry['wav_bytes'], content_type='audio/wav')
    result = fal_client.subscribe(
        'nvidia/nemotron-3-nano-omni/audio',
        arguments={
            'prompt': question,
            'audio_url': audio_url,
            'reasoning_mode': 'no_think',
            'max_tokens': 300,
            'temperature': 0.7,
        },
    )
    return result.get('output', str(result)).strip()


def call_nemotron(entry, question):
    try:
        return _nemotron_call(entry, question)
    except Exception as e:
        return f'ERROR: {e}'


MODELS = {
    'gemini':    {'caller': call_gemini,    'label': 'Gemini 2.5 Flash genai 2.0.0'},
    'gpt_audio': {'caller': call_gpt_audio, 'label': 'gpt-audio 2025-08-28'},
    'nemotron':  {'caller': call_nemotron,  'label': 'Nemotron 3 Nano Omni'},
}

print('Models:', {mk: mv['label'] for mk, mv in MODELS.items()})

## 6. Quick test with one child across all models

In [ ]:
test = manifest[0]
test_q = QUESTIONS['q3_age']['text']
print(f'Testing: {test["child"]} (age {test["age"]}, {test["duration"]:.1f}s)\n')

for mk, info in MODELS.items():
    t0 = time.time()
    resp = info['caller'](test, test_q)
    dt = time.time() - t0
    status = 'ERROR' if resp.startswith('ERROR') else 'OK'
    print(f'{info["label"]:.<32s} {status} ({dt:.1f}s)')
    print(f'  {resp[:200]}')
    print()

## 7. Run full experiment

This is 756 model calls and takes roughly 30 to 60 minutes.

In [ ]:
from tqdm.auto import tqdm

RESUME_FROM = 'study1b_pinned_raw_results.csv'   # the file you already saved
CHECKPOINT = 'study1b_pinned_raw_results.partial.csv'

# Load prior results if present, otherwise start clean.
try:
    prior = pd.read_csv(RESUME_FROM)
    results = prior.to_dict('records')
    print(f'Loaded {len(results)} prior results from {RESUME_FROM}')
except FileNotFoundError:
    results = []
    print(f'{RESUME_FROM} not found, starting fresh')

# A call is identified by child, model, question, and repeat number.
done_keys = {(r['child'], r['model'], r['question'], int(r['repeat'])) for r in results}

all_jobs = [(entry, mk, info, qk, qi, rep)
            for entry in manifest
            for mk, info in MODELS.items()
            for qk, qi in QUESTIONS.items()
            for rep in range(N_REPEATS)]

# Keep only jobs whose key is not already present.
jobs = [j for j in all_jobs
        if (j[0]['child'], j[1], j[3], j[5] + 1) not in done_keys]

total_all = len(all_jobs)
total = len(jobs)
n_err = 0
n_ref = 0
t0 = time.time()

print(f'{total} calls remaining out of {total_all} total.', flush=True)

bar = tqdm(jobs, total=total, desc='API calls', unit='call')
try:
    for entry, mk, info, qk, qi, rep in bar:
        resp = info['caller'](entry, qi['text'])
        if resp.startswith('ERROR'):
            n_err += 1
        elif re.search(r"sorry|can't", resp, re.I):
            n_ref += 1
        results.append({
            'child': entry['child'], 'actual_age': entry['age'],
            'gender': entry['gender'], 'duration': entry['duration'],
            'model': mk, 'model_label': info['label'],
            'question': qk, 'question_label': qi['label'],
            'repeat': rep + 1, 'response': resp,
        })
        done_new = len(results) - len(done_keys)
        # Flush all results so far to disk so an interrupt or crash never loses progress.
        if done_new % 25 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT, index=False)
        elapsed = time.time() - t0
        rate = done_new / elapsed if elapsed > 0 else 0
        eta_min = (total - done_new) / rate / 60 if rate > 0 else 0
        bar.set_postfix_str(
            f'{done_new}/{total} new, eta {eta_min:.1f}m, '
            f'errors {n_err}, refusals {n_ref}, now {entry["child"]}/{mk}')
        time.sleep(RATE_LIMIT_DELAY)
finally:
    bar.close()
    results_df = pd.DataFrame(results)
    if len(results_df) > 0:
        results_df.to_csv(CHECKPOINT, index=False)
    print(f'\nStopped with {len(results_df)} of {total_all} total results. '
          f'New this run {len(results_df) - len(done_keys)}, '
          f'errors {n_err}, refusals {n_ref}. Checkpoint saved to {CHECKPOINT}.', flush=True)

## 8. Diagnostics

In [ ]:
for mk in MODELS:
    sub = results_df[results_df['model'] == mk]
    errs = sub[sub['response'].str.startswith('ERROR')]
    refs = sub[sub['response'].str.contains(
        "sorry|can't estimate|can't determine|can't identify|can't analyze",
        case=False, na=False)]
    ok = sub[~sub.index.isin(errs.index) & ~sub.index.isin(refs.index)]
    print(f'{MODELS[mk]["label"]:.<32s} Errors:{len(errs):>4d}  '
          f'Refusals:{len(refs):>4d}  Answers:{len(ok):>4d}')
    if len(errs) > 0:
        print(f'  Error: {errs.iloc[0]["response"][:200]}')
    if len(refs) > 0:
        print(f'  Refusal: {refs.iloc[0]["response"][:200]}')

## 9. gpt-audio refusal analysis

In [ ]:
gpt_q3 = results_df[(results_df['model'] == 'gpt_audio') & (results_df['question'] == 'q3_age')].copy()
gpt_q3['refused'] = (
    gpt_q3['response'].str.contains(
        "sorry|can't estimate|can't determine|can't identify|can't analyze",
        case=False, na=False) |
    gpt_q3['response'].str.startswith('ERROR'))

print('gpt-audio refusal rate by age')
for age in sorted(gpt_q3['actual_age'].unique()):
    s = gpt_q3[gpt_q3['actual_age'] == age]
    print(f'  Age {age}: {100*s["refused"].mean():.0f}% refused (n={s["child"].nunique()})')

cr = gpt_q3.groupby(['child','actual_age'])['refused'].mean().reset_index()
if cr['refused'].nunique() > 1 and len(cr) > 2:
    r, p = stats.pearsonr(cr['actual_age'], cr['refused'])
    print(f'\nAge-refusal correlation: r={r:.3f}, p={p:.4f}')
else:
    print('\nRefusal is uniform across ages, so no age correlation is computed.')

print(f'\nOverall gpt-audio Q3 refusal rate: {100*gpt_q3["refused"].mean():.0f}%')

## 10. Parse responses

In [ ]:
def parse_yn(r):
    if r.startswith('ERROR'): return 'ERROR'
    if re.search(r"sorry|can't", r, re.I): return 'REFUSED'
    u = r.strip().upper()
    if u.startswith('YES'): return 'YES'
    if u.startswith('NO'): return 'NO'
    fl = u.split('\n')[0]
    if 'YES' in fl and 'NO' not in fl: return 'YES'
    if 'NO' in fl and 'YES' not in fl: return 'NO'
    return 'UNCLEAR'

def parse_age(r):
    if r.startswith('ERROR'): return None
    if re.search(r"sorry|can't estimate|can't determine|can't identify|can't analyze", r, re.I):
        return None
    m = re.search(r'[Ee]stimated\s+[Aa]ge[:\s]+(\d+)', r)
    if m: return int(m.group(1))
    m = re.search(r'(\d+)\s*-\s*(\d+)\s*years', r)
    if m: return (int(m.group(1)) + int(m.group(2))) / 2
    nums = re.findall(r'\d+', r.split('\n')[0])
    return int(nums[0]) if nums else None

q1 = results_df[results_df['question'] == 'q1_over_13'].copy()
q1['pred'] = q1['response'].apply(parse_yn)
q2 = results_df[results_df['question'] == 'q2_over_18'].copy()
q2['pred'] = q2['response'].apply(parse_yn)
q3 = results_df[results_df['question'] == 'q3_age'].copy()
q3['pred_age'] = q3['response'].apply(parse_age)
q3['signed_error'] = q3['pred_age'] - q3['actual_age']
valid_q3 = q3.dropna(subset=['pred_age'])

print('Valid Q3:')
for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    print(f'  {MODELS[mk]["label"]}: {len(s)} from {s["child"].nunique()} children')

## 11. Binary classification on Q1 and Q2

In [ ]:
for label, qdf in [("Q1: Over 13? (correct: NO)", q1), ("Q2: Over 18? (correct: NO)", q2)]:
    print(label)
    print('=' * 60)
    for mk in MODELS:
        s = qdf[(qdf['model'] == mk) & qdf['pred'].isin(['YES','NO'])]
        if len(s) == 0:
            print(f'  {MODELS[mk]["label"]}: no valid responses'); continue
        print(f'  {MODELS[mk]["label"]}:')
        for age in sorted(s['actual_age'].unique()):
            a = s[s['actual_age'] == age]
            print(f'    Age {age} (n={a["child"].nunique()}): {100*(a["pred"]=="YES").mean():.0f}% YES')
    print()

## 12. Age estimation on Q3

### 12a. Overall bias

In [ ]:
print('OVERALL BIAS')
print('=' * 60)

for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    if len(s) == 0:
        print(f'{MODELS[mk]["label"]}: no valid predictions'); continue
    e = s['signed_error']
    cm = s.groupby('child')['signed_error'].mean()
    print(f'\n{MODELS[mk]["label"]} ({s["child"].nunique()} children):')
    print(f'  Mean signed error: {e.mean():+.2f} years')
    print(f'  Median: {e.median():+.1f} | MAE: {e.abs().mean():.2f}')
    if len(cm) >= 3:
        t, p = stats.ttest_1samp(cm, 0)
        sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
        print(f'  t={t:.3f}, p={p:.4f} {sig}')

### 12b. By age

In [ ]:
for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    if len(s) == 0: continue
    print(f'{MODELS[mk]["label"]}:')
    print(f'  {"Age":>4s} {"n":>3s} {"Est":>6s} {"Err":>6s}')
    for age in sorted(s['actual_age'].unique()):
        a = s[s['actual_age'] == age]
        print(f'  {age:>4d} {a["child"].nunique():>3d} {a["pred_age"].mean():>6.1f} {a["signed_error"].mean():>+6.1f}')
    print()

### 12c. Compression test

In [ ]:
for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    cm = s.groupby('child').agg(age=('actual_age','first'), err=('signed_error','mean'))
    if len(cm) < 3:
        print(f'{MODELS[mk]["label"]}: skipping ({len(cm)} children)'); continue
    r, p = stats.pearsonr(cm['age'], cm['err'])
    sig = '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else 'n.s.'
    print(f'{MODELS[mk]["label"]}: r={r:.3f}, p={p:.4f} {sig}')
    print(f'  {"=> Compression" if r < 0 else "=> No compression"}')

### 12d. Per-child detail

In [ ]:
active = [mk for mk in MODELS if len(valid_q3[valid_q3['model']==mk]) > 0]
header = f'{"Child":>7s} {"Age":>4s} {"G":>2s}'
for mk in active: header += f' {MODELS[mk]["label"][:11]:>11s}'
print(header)
print('-' * len(header))
for e in sorted(manifest, key=lambda x: (x['age'], x['child'])):
    line = f'{e["child"]:>7s} {e["age"]:>4d} {e["gender"]:>2s}'
    for mk in active:
        s = valid_q3[(valid_q3['model']==mk) & (valid_q3['child']==e['child'])]
        if len(s) > 0:
            line += f' {s["pred_age"].mean():>5.1f}({s["signed_error"].mean():+4.1f})'
        else:
            line += f' {"refused":>11s}'
    print(line)

## 13. Visualization

### 13a. Predicted vs actual

In [ ]:
active = [mk for mk in MODELS if len(valid_q3[valid_q3['model']==mk]) > 0]
colors = {'gemini':'steelblue','gpt_audio':'coral','nemotron':'seagreen'}
fig, axes = plt.subplots(1, len(active), figsize=(7*len(active), 5))
if len(active)==1: axes=[axes]
for i, mk in enumerate(active):
    ax = axes[i]
    s = valid_q3[valid_q3['model']==mk]
    cs = s.groupby('child').agg(age=('actual_age','first'), est=('pred_age','mean')).reset_index()
    j = np.random.default_rng(42).normal(0, 0.1, len(cs))
    ax.scatter(cs['age']+j, cs['est'], c=colors.get(mk,'gray'), s=80, edgecolor='white', alpha=.8)
    ax.plot([3,50],[3,50],'k--',lw=1,alpha=.4)
    ax.set_xlabel('Actual Age'); ax.set_ylabel('Predicted Age')
    ax.set_title(f'{MODELS[mk]["label"]} ({len(cs)} children)')
    ax.set_xlim(3,13)
sns.despine(); plt.tight_layout()
plt.savefig('study1b_pinned_pred_vs_actual.png', dpi=150, bbox_inches='tight'); plt.show()

### 13b. Error by age

In [ ]:
fig, axes = plt.subplots(1, len(active), figsize=(7*len(active), 5))
if len(active)==1: axes=[axes]
for i, mk in enumerate(active):
    ax = axes[i]
    s = valid_q3[valid_q3['model']==mk]
    ages = sorted(s['actual_age'].unique())
    bp = ax.boxplot([s[s['actual_age']==a]['signed_error'].values for a in ages],
                     tick_labels=ages, patch_artist=True, medianprops={'color':'black','linewidth':1.5})
    for p in bp['boxes']: p.set_facecolor(colors.get(mk,'gray')); p.set_alpha(.6)
    ax.axhline(y=0,color='black',linestyle='--',lw=1,alpha=.5)
    ax.set_xlabel('Actual Age'); ax.set_ylabel('Signed Error')
    ax.set_title(MODELS[mk]['label'])
sns.despine(); plt.tight_layout()
plt.savefig('study1b_pinned_error_by_age.png', dpi=150, bbox_inches='tight'); plt.show()

### 13c. Model comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ages = sorted(valid_q3['actual_age'].unique())
x = np.arange(len(ages))
w = 0.8 / len(active)
for i, mk in enumerate(active):
    s = valid_q3[valid_q3['model']==mk]
    means = [s[s['actual_age']==a]['signed_error'].mean() if len(s[s['actual_age']==a])>0 else 0 for a in ages]
    sems = [s[s['actual_age']==a]['signed_error'].sem() if len(s[s['actual_age']==a])>1 else 0 for a in ages]
    ax.bar(x+(i-len(active)/2+.5)*w, means, w, yerr=sems, label=MODELS[mk]['label'],
           color=colors.get(mk,'gray'), alpha=.7, edgecolor='white', capsize=3)
ax.axhline(y=0,color='black',linestyle='--',lw=1,alpha=.5)
ax.set_xticks(x); ax.set_xticklabels(ages)
ax.set_xlabel('Actual Age'); ax.set_ylabel('Mean Signed Error')
ax.set_title('Study 1b rerun, age estimation error on spontaneous speech'); ax.legend()
sns.despine(); plt.tight_layout()
plt.savefig('study1b_pinned_model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

### 13d. gpt-audio refusal rate by age

In [ ]:
gpt_all = results_df[results_df['model']=='gpt_audio'].copy()
gpt_all['refused'] = (gpt_all['response'].str.contains(
    "sorry|can't estimate|can't determine|can't identify|can't analyze",
    case=False,na=False) | gpt_all['response'].str.startswith('ERROR'))
rba = gpt_all.groupby('actual_age')['refused'].mean()
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(rba.index, rba.values*100, color='coral', alpha=.7, edgecolor='white')
ax.set_xlabel('Actual Age'); ax.set_ylabel('Refusal Rate (%)')
ax.set_title('gpt-audio refusal rate by age, spontaneous speech'); ax.set_ylim(0,105)
sns.despine(); plt.tight_layout()
plt.savefig('study1b_pinned_gpt_audio_refusal.png', dpi=150, bbox_inches='tight'); plt.show()

## 14. Run summary

In [ ]:
print('STUDY 1b RERUN SUMMARY')
print('Spontaneous speech, pinned dependencies')
print('=' * 60)
for mk in MODELS:
    s = valid_q3[valid_q3['model'] == mk]
    err_total = results_df[results_df['model'] == mk]
    errs = err_total[err_total['response'].str.startswith('ERROR')]
    refs = err_total[err_total['response'].str.contains("sorry|can't", case=False, na=False)]
    if len(s) == 0:
        print(f'  {MODELS[mk]["label"]}: no valid predictions, '
              f'refusal rate={100*len(refs)/max(len(err_total),1):.0f}%')
        continue
    print(f'  {MODELS[mk]["label"]}: mean estimate={s["pred_age"].mean():.1f}, '
          f'MAE={s["signed_error"].abs().mean():.1f}, '
          f'refusal rate={100*len(refs)/len(err_total):.0f}%')

## 15. Sample reasoning

In [ ]:
for mk in active:
    print(f'\n{"="*60}\n{MODELS[mk]["label"]}\n{"="*60}')
    for ta in [5, 7, 10]:
        s = q3[(q3['model']==mk) & (q3['actual_age']==ta)]
        ans = s[s['pred_age'].notna()]
        if len(ans)>0:
            r = ans.iloc[0]
            print(f'\n  [{r["child"]}, age {ta}]: {r["response"][:300]}')
        else:
            ref = s[s['response'].str.contains("sorry|can't",case=False,na=False)]
            if len(ref)>0:
                print(f'\n  [age {ta}, REFUSED]: {ref.iloc[0]["response"][:200]}')

## 16. Save

In [ ]:
results_df.to_csv('study1b_pinned_raw_results.csv', index=False)
valid_q3.to_csv('study1b_pinned_q3_parsed.csv', index=False)
summary = valid_q3.groupby(['child','actual_age','gender','model']).agg(
    mean_pred=('pred_age','mean'), std_pred=('pred_age','std'),
    mean_error=('signed_error','mean'), n=('pred_age','count')).reset_index()
summary.to_csv('study1b_pinned_summary.csv', index=False)
print('Saved: study1b_pinned_raw_results.csv, study1b_pinned_q3_parsed.csv, study1b_pinned_summary.csv')

## 17. Conclusion

This run scored 28 children aged 5 to 10 on spontaneous speech across three audio models under fully pinned dependencies. Each child was asked whether the speaker is over 13, whether the speaker is over 18, and to estimate the speaker's age, with three repeats per question. We test the hypothesis that spontaneous speech will produce different age prediction estimates relative to scripted speech for the children in the Kids VA coprus.

- Here again, Gemini 2.5 Flash under `google-genai` version `2.0.0` produced accurate and roughly unbiased estimates. Its mean signed error was about half a year below the true age and its mean absolute error was about 1.5 years. It refused nothing.
- The OpenAI model again declined the task. It refused about 75% of age questions and returned no usable age estimates, so it has no scored predictions here.
- Nemotron 3 Nano Omni answered across the full age range but overestimated. Its mean signed error was about plus 2.6 years and its mean absolute error was about 3.6 years, and a few children drew very large overestimates. On spontaneous speech Nemotron reverses its behaviro with scripted behavior, overestimating ages by about 2.6 years overall and nailing the oldest children rather than reading them young, which undercuts the idea that its scripted underestimation reflected a stable COPPA-driven reluctance to age children past the threshold.

Because every package version is pinned, these differences reflect model behavior rather than environment drift.